# Примеры: кодирования категорий

## One-Hot Encoding

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Создаём DataFrame
data = {
    'день недели': ['понедельник', 'вторник', 'среда', 'четверг', 'пятница'],
    'погода': ['солнечно', 'облачно', 'дождливо', 'солнечно', 'облачно'],
    'температура': ['тёплая', 'прохладная', 'холодная', 'тёплая', 'прохладная']
}
df = pd.DataFrame(data)

# Создаём экземпляр класса OneHotEncoder
# sparse=False - выводит объекм матрицей. По умолчанию он в виде разряженной матрицы. Это более эффективно, но его не посмотреть
# handle_unknown - как поступить с категориями, которых нет в обучении
# handle_unknown='error'
# handle_unknown='ignore'
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Применяем OneHotEncoder к столбцам "погода" и "температура"
encoder.fit(df[['погода', 'температура']])
encoded_columns = encoder.transform(df[['погода', 'температура']])

print(encoded_columns)

[[0. 0. 1. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0.]]


### Кодирование признаков

Кодирование состоит из двух действий:

Обучение кодировщика (метод fit() ): этот шаг похож на обучение модели: кодировщик изучает данные и определяет, сколько и какие новые колонки нужно создать.

Преобразование признаков по выявленным правилам (метод transform() ): на этом шаге кодировщик возвращает данные с новыми колонками.

### Замена исходных признаков 

Признаки успешно закодированы, однако пока что они хранятся в переменной `encoded_columns` в виде двухмерного массива. Нужно преобразовать этот массив в новый датафрейм и объединить его с исходной выборкой, удалив из неё старые версии колонок.

Задание

- Создайте новый датасет с закодированными столбцами. Чтобы получить названия новых колонок, используйте метод `encoder.get_feature_names_out(['погода', 'температура'])`. Названия колонок генерируются по шаблону `<исходный признак>_<категория>` (`погода_солнечно, погода_облачно` и т.д.).

- Присоедините новый датасет к старому с помощью метода `pd.concat` и удалите из получившейся таблицы категориальные признаки, которые больше не нужны.

In [2]:
# Создайте новый DataFrame с закодированными столбцами
encoded_df = pd.DataFrame(
    encoded_columns, 
    columns=encoder.get_feature_names_out(['погода', 'температура'])
)

# Объедините исходную таблицу с новыми столбцами
df_encoded = pd.concat([df, encoded_df], axis=1)

# Удалите исходные категориальные столбцы
df_encoded = df_encoded.drop(columns=['погода', 'температура'])

print(df_encoded)

   день недели  погода_дождливо  погода_облачно  погода_солнечно  \
0  понедельник              0.0             0.0              1.0   
1      вторник              0.0             1.0              0.0   
2        среда              1.0             0.0              0.0   
3      четверг              0.0             0.0              1.0   
4      пятница              0.0             1.0              0.0   

   температура_прохладная  температура_тёплая  температура_холодная  
0                     0.0                 1.0                   0.0  
1                     1.0                 0.0                   0.0  
2                     0.0                 0.0                   1.0  
3                     0.0                 1.0                   0.0  
4                     1.0                 0.0                   0.0  


#### Примеры кодирования на других данных


Вы работаете в банке: по характеристикам клиента нужно предсказать вероятность того, сможет ли он выплатить кредит. Характеристики представлены категориальными признаками, и перед моделированием их нужно закодировать.

В датасете с клиентами есть три категориальных признака: Профессия, Семейное положение и Тип жилья. В каждом из них значения представлены строками с названиями категорий.

In [4]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

data = {
    'Имя': ['Анна', 'Иван', 'Мария', 'Сергей', 'Ольга', 'Дмитрий', 'Елена', 'Павел'],
    'Профессия': ['Врач', 'Учитель', 'Инженер', 'Врач', 'Инженер', 'Учитель', 'Учитель', 'Инженер'],
    'Семейное положение': ['Замужем', 'Холост', 'В разводе', 'Холост', 'Замужем', 'В разводе', 'Замужем', 'Холост'],
    'Тип жилья': ['Собственная', 'Съемная', 'С родителями', 'Съемная', 'Собственная', 'С родителями', 'Съемная', 'Собственная']
}
users_df = pd.DataFrame(data)

# отдельно вынесем колонки для кодирования 
columns_to_encode = ['Профессия', 'Семейное положение', 'Тип жилья']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Используем fit_transform для обучения и трансформации за один шаг
encoded_columns = encoder.fit_transform(users_df[columns_to_encode])

encoded_df = pd.DataFrame(
    encoded_columns,
    columns=encoder.get_feature_names_out(columns_to_encode) # Получили все новые названия колонок
)

# Объедините исходную таблицу с новыми столбцами
users_df_encoded = pd.concat([users_df, encoded_df], axis=1)

# Удалите исходные категориальные столбцы
users_df_encoded = users_df_encoded.drop(columns=columns_to_encode)

# Результат
print(users_df_encoded)


       Имя  Профессия_Врач  Профессия_Инженер  Профессия_Учитель  \
0     Анна             1.0                0.0                0.0   
1     Иван             0.0                0.0                1.0   
2    Мария             0.0                1.0                0.0   
3   Сергей             1.0                0.0                0.0   
4    Ольга             0.0                1.0                0.0   
5  Дмитрий             0.0                0.0                1.0   
6    Елена             0.0                0.0                1.0   
7    Павел             0.0                1.0                0.0   

   Семейное положение_В разводе  Семейное положение_Замужем  \
0                           0.0                         1.0   
1                           0.0                         0.0   
2                           1.0                         0.0   
3                           0.0                         0.0   
4                           0.0                         1.0   
5        

## Target Encoding 

Поскольку Target Encoding использует информацию о целевой переменной, нельзя кодировать признаки сразу на всём датасете. Прежде чем переходить к кодированию, необходимо разделить данные на обучающую и тестовую выборки.

Почему это важно?

- Если посчитать средние значения по всей таблице, информация из тестовой выборки повлияет на закодированные значения, на которых будет обучаться модель.
- Модель будет знать ответы наперёд и, скорее всего, переобучится.

In [5]:
from sklearn.model_selection import train_test_split

# Создаём новый DataFrame. На этот раз - с колонкой 'завтра дождь'
data = {
    'день недели': ['понедельник', 'вторник', 'среда', 'четверг', 'пятница'],
    'погода': ['солнечно', 'облачно', 'дождливо', 'солнечно', 'облачно'],
    'температура': ['тёплая', 'прохладная', 'холодная', 'тёплая', 'прохладная'],
    'завтра дождь': [0, 1, 1, 0, 1]
}
df = pd.DataFrame(data)

# Разделяем данные
X = df[['день недели', 'погода', 'температура']]
y = df['завтра дождь']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=627)

Теперь можно применять TargetEncoder для преобразования. Как и OneHotEncoder, это класс, который нужно импортировать, а затем использовать для создания объекта-кодировщика.

In [7]:
from category_encoders import TargetEncoder

# Создаём TargetEncoder
encoder = TargetEncoder()

# Кодируем только на train
X_train_encoded = encoder.fit_transform(X_train, y_train)

# Применяем к тестовым данным
X_test_encoded = encoder.transform(X_test)

print("Обучающая выборка после кодирования:")
print(X_train_encoded)

print("Тестовая выборка после кодирования:")
print(X_test_encoded)

Обучающая выборка после кодирования:
   день недели    погода  температура
3     0.579928  0.579928     0.579928
2     0.710036  0.710036     0.710036
4     0.710036  0.710036     0.710036
Тестовая выборка после кодирования:
   день недели    погода  температура
0     0.666667  0.579928     0.579928
1     0.666667  0.710036     0.710036


В обучающей выборке категориальные признаки заменились на числовые значения — средние значения целевой переменной для каждой категории. Благодаря разделению в тестовой выборке категории закодировались по тем же правилам, которые были выучены на train. Например, солнечная погода получила значение 0.579928 — и именно оно используется в обеих таблицах.

### Задание 2

One-Hot кодирование помогло преобразовать текстовые характеристики клиентов банка, однако нам бы хотелось, чтобы при обучении модели учитывалась связь целевой переменной с каждой категорией. Чтобы этого добиться, закодируем данные с помощью Target Encoding.

Напишите код для преобразования признаков Профессия, Семейное положение и Тип жилья в числа с учётом значения целевого признака Вернул кредит. В этом вам поможет TargetEncoder.

Шаги решения:
- Разделите данные на обучающие и тестовые так , чтобы в тестовой выборке было 20% записей. При делении задайте параметр random_state = 627.
- Преобразуйте данные в обеих выборках и сохраните результат в переменные X_train_encoded и X_test_encoded.

In [ ]:
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

data = {
    'Имя': ['Анна', 'Иван', 'Мария', 'Сергей', 'Ольга', 'Дмитрий', 'Елена', 'Павел'],
    'Профессия': ['Врач', 'Учитель', 'Инженер', 'Врач', 'Инженер', 'Учитель', 'Учитель', 'Инженер'],
    'Семейное положение': ['Замужем', 'Холост', 'В разводе', 'Холост', 'Замужем', 'В разводе', 'Замужем', 'Холост'],
    'Тип жилья': ['Собственная', 'Съемная', 'С родителями', 'Съемная', 'Собственная', 'С родителями', 'Съемная', 'Собственная'],
    'Вернул кредит': [0, 1, 1, 0, 1, 0, 0, 1]
}
users_df = pd.DataFrame(data)

# Сохраните названия колонок для кодирования и целевую переменную
columns_to_encode = ['Профессия', 'Семейное положение', 'Тип жилья']
target = 'Вернул кредит'

# Ваш код здесь: поделите и преобразуйте данные
X = users_df[columns_to_encode]
y = users_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.2, random_state=627)

encoder = TargetEncoder()
# обучаем разделять категории
X_train_encoded = encoder.fit_transform(X_train, y_train)

# Применяем к тестовым данным
X_test_encoded = encoder.transform(X_test)

print("Обучающая выборка после кодирования:")
print(X_train_encoded)

print("Тестовая выборка после кодирования:")
print(X_test_encoded)

Обучающая выборка после кодирования:
        Имя  Профессия  Семейное положение  Тип жилья
3  0.434946   0.434946                 0.5   0.474256
2  0.565054   0.570926                 0.5   0.500000
1  0.565054   0.474256                 0.5   0.474256
5  0.434946   0.474256                 0.5   0.500000
6  0.434946   0.474256                 0.5   0.474256
4  0.565054   0.570926                 0.5   0.565054
Тестовая выборка после кодирования:
   Имя  Профессия  Семейное положение  Тип жилья
0  0.5   0.434946                 0.5   0.565054
7  0.5   0.570926                 0.5   0.565054
